In [1]:
# ============================================================
# Question 1a: Baseline Reproduction (Decision Trees per class)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# ------------------------------
# 1. Paths and feature keys
# ------------------------------
base_path = Path("D:\JKU\Semester 3\PC\Task 5")
train_path = base_path / "train"
val_path = base_path / "validation"

FEATURE_KEYS = [
    'zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max',
    'melspect_mean', 'melspect_std', 'melspect_min', 'melspect_max',
    'mfcc_mean', 'mfcc_std', 'mfcc_min', 'mfcc_max',
    'mfcc_d_mean', 'mfcc_d_std', 'mfcc_d_min', 'mfcc_d_max',
    'mfcc_d2_mean', 'mfcc_d2_std', 'mfcc_d2_min', 'mfcc_d2_max',
    'flux_mean', 'flux_std', 'flux_min', 'flux_max',
    'flatness_mean', 'flatness_std', 'flatness_min', 'flatness_max',
    'centroid_mean', 'centroid_std', 'centroid_min', 'centroid_max',
    'bandwidth_mean', 'bandwidth_std', 'bandwidth_min', 'bandwidth_max',
    'contrast_mean', 'contrast_std', 'contrast_min', 'contrast_max',
    'rolloff_low_mean', 'rolloff_low_std', 'rolloff_low_min', 'rolloff_low_max',
    'rolloff_high_mean', 'rolloff_high_std', 'rolloff_high_min', 'rolloff_high_max',
    'energy_mean', 'energy_std', 'energy_min', 'energy_max',
    'power_mean', 'power_std', 'power_min', 'power_max',
]

# ------------------------------
# 2. Helper functions
# ------------------------------
def load_features_labels(npz_files, feature_keys):
    X_list, y_list = [], []
    class_names = None
    valid_files = []
    
    # First filter: keep only files with >= 2 annotators
    for f in npz_files:
        data = np.load(f, allow_pickle=True)
        if data['annotations'].shape[2] >= 2:
            valid_files.append(f)
    
    print(f"Files with ≥2 annotators: {len(valid_files)} out of {len(npz_files)}")
    
    for f in valid_files:
        data = np.load(f, allow_pickle=True)
        if class_names is None:
            class_names = data['class_names']
        # Aggregate annotations: majority vote (>= half annotators)
        ann = data['annotations']
        A = ann.shape[2]
        binary = (ann >= 0.5).astype(int)
        labels = (binary.sum(axis=2) > A/2).astype(int)
        # Build feature vector
        feats = []
        for key in feature_keys:
            feat = data[key]
            if feat.ndim > 1:
                feats.append(feat.reshape(feat.shape[0], -1))
            else:
                feats.append(feat.reshape(-1, 1))
        X = np.hstack(feats)
        X_list.append(X)
        y_list.append(labels)
    
    return np.vstack(X_list), np.vstack(y_list), class_names

# ------------------------------
# 3. Load files
# ------------------------------
train_files = list((train_path / "audio_features").glob("*.npz"))
val_files = list((val_path / "audio_features").glob("*.npz"))

print(f"Train files: {len(train_files)}, Validation files: {len(val_files)}")

X_train, y_train, class_names = load_features_labels(train_files, FEATURE_KEYS)
X_val, y_val, _ = load_features_labels(val_files, FEATURE_KEYS)

# Split validation into validation (50%) and non‑hidden test (50%)
X_val_split, X_test, y_val_split, y_test = train_test_split(
    X_val, y_val, test_size=0.5, random_state=42, stratify=None
)

print(f"Training segments: {X_train.shape[0]}")
print(f"Validation segments: {X_val_split.shape[0]}")
print(f"Test (non‑hidden) segments: {X_test.shape[0]}")

# ------------------------------
# 4. Preprocessing (StandardScaler + PCA)
# ------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val_split)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA components: {X_train_pca.shape[1]}")

# ------------------------------
# 5. Train one DecisionTreeClassifier per class (baseline)
# ------------------------------
n_classes = len(class_names)
baseline_models = []

for c in range(n_classes):
    print(f"Training classifier for class {c+1}/{n_classes}: {class_names[c]}...")
    clf = DecisionTreeClassifier(max_depth=10, random_state=42)   # limit depth for speed
    clf.fit(X_train_pca, y_train[:, c])
    baseline_models.append(clf)
    print(f"   Done.")

# ------------------------------
# 6. Evaluate on non‑hidden test set (segment‑based macro F1)
# ------------------------------
y_pred_test = np.zeros_like(y_test)
for c, clf in enumerate(baseline_models):
    y_pred_test[:, c] = clf.predict(X_test_pca)

# Compute per‑class F1, then macro F1
per_class_f1 = []
for c in range(n_classes):
    f1 = f1_score(y_test[:, c], y_pred_test[:, c], zero_division=0)
    per_class_f1.append(f1)

macro_f1 = np.mean(per_class_f1)

print("\n" + "="*50)
print("Baseline (Decision Trees) Results on Non‑hidden Test Set")
print("="*50)
for i, name in enumerate(class_names):
    print(f"{name:30s} F1 = {per_class_f1[i]:.4f}")
print(f"\nSegment‑based Macro F1 Score: {macro_f1:.4f}")

Train files: 3704, Validation files: 999
Files with ≥2 annotators: 3457 out of 3704
Files with ≥2 annotators: 922 out of 999
Training segments: 159084
Validation segments: 21601
Test (non‑hidden) segments: 21602
PCA components: 239
Training classifier for class 1/15: bell_ringing...
   Done.
Training classifier for class 2/15: coffee_machine...
   Done.
Training classifier for class 3/15: cutlery_dishes...
   Done.
Training classifier for class 4/15: door_open_close...
   Done.
Training classifier for class 5/15: footsteps...
   Done.
Training classifier for class 6/15: keyboard_typing...
   Done.
Training classifier for class 7/15: keychain...
   Done.
Training classifier for class 8/15: light_switch...
   Done.
Training classifier for class 9/15: microwave...
   Done.
Training classifier for class 10/15: phone_ringing...
   Done.
Training classifier for class 11/15: running_water...
   Done.
Training classifier for class 12/15: toilet_flushing...
   Done.
Training classifier for clas

In [2]:
# ============================================================
# Question 2a: Train Best Classifier (Random Forest)
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

# ------------------------------
# Use the same data splits from baseline (already loaded)
# X_train_pca, y_train, X_test_pca, y_test are already defined
# ------------------------------

# ------------------------------
# Train Random Forest (one per class) using MultiOutputClassifier
# ------------------------------
print("\n" + "="*60)
print("Training Random Forest (class_weight='balanced')")
print("="*60)

# Hyperparameters (from Task 4 best model)
rf_params = {
    'n_estimators': 200,
    'max_depth': 10,
    'min_samples_leaf': 5,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

print("Hyperparameters:")
for k, v in rf_params.items():
    print(f"  {k}: {v}")

rf_model = MultiOutputClassifier(
    RandomForestClassifier(**rf_params)
)

rf_model.fit(X_train_pca, y_train)

# ------------------------------
# Evaluate on non‑hidden test set
# ------------------------------
y_pred_test = rf_model.predict(X_test_pca)

per_class_f1_rf = []
for c in range(len(class_names)):
    f1 = f1_score(y_test[:, c], y_pred_test[:, c], zero_division=0)
    per_class_f1_rf.append(f1)

macro_f1_rf = np.mean(per_class_f1_rf)

print("\n" + "="*50)
print("Random Forest Results on Non‑hidden Test Set")
print("="*50)
for i, name in enumerate(class_names):
    print(f"{name:30s} F1 = {per_class_f1_rf[i]:.4f}")
print(f"\nSegment‑based Macro F1 Score: {macro_f1_rf:.4f}")

# ------------------------------
# Compare with baseline (optional, for 2c)
# ------------------------------
print("\n" + "="*50)
print("Comparison: Baseline (Decision Trees) vs Random Forest")
print("="*50)
print(f"Baseline Macro F1: 0.3038")
print(f"Random Forest Macro F1: {macro_f1_rf:.4f}")
print(f"Improvement: {macro_f1_rf - 0.3038:.4f}")



Training Random Forest (class_weight='balanced')
Hyperparameters:
  n_estimators: 200
  max_depth: 10
  min_samples_leaf: 5
  class_weight: balanced
  random_state: 42
  n_jobs: -1

Random Forest Results on Non‑hidden Test Set
bell_ringing                   F1 = 0.3741
coffee_machine                 F1 = 0.4657
cutlery_dishes                 F1 = 0.4044
door_open_close                F1 = 0.3069
footsteps                      F1 = 0.4659
keyboard_typing                F1 = 0.5368
keychain                       F1 = 0.4461
light_switch                   F1 = 0.1856
microwave                      F1 = 0.5785
phone_ringing                  F1 = 0.5873
running_water                  F1 = 0.7260
toilet_flushing                F1 = 0.3651
vacuum_cleaner                 F1 = 0.5914
wardrobe_drawer_open_close     F1 = 0.2189
window_open_close              F1 = 0.1677

Segment‑based Macro F1 Score: 0.4280

Comparison: Baseline (Decision Trees) vs Random Forest
Baseline Macro F1: 0.3038
Random 

In [ ]:
# ============================================================
# Question 2b: Hyperparameter Tuning (Random Forest)
# ============================================================

import time
import matplotlib.pyplot as plt

# Define hyperparameter grid
max_depth_values = [5, 10, 15, 20]
n_estimators_values = [50, 100, 200]

results = []

for max_depth in max_depth_values:
    for n_est in n_estimators_values:
        print(f"Training: max_depth={max_depth}, n_estimators={n_est}...")
        start = time.time()
        
        rf = MultiOutputClassifier(
            RandomForestClassifier(
                n_estimators=n_est,
                max_depth=max_depth,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )
        )
        rf.fit(X_train_pca, y_train)
        y_pred = rf.predict(X_test_pca)
        
        # Compute Macro F1
        f1s = []
        for c in range(len(class_names)):
            f1s.append(f1_score(y_test[:, c], y_pred[:, c], zero_division=0))
        macro_f1 = np.mean(f1s)
        
        results.append({
            'max_depth': max_depth,
            'n_estimators': n_est,
            'macro_f1': macro_f1,
            'time': time.time() - start
        })
        print(f"   Macro F1 = {macro_f1:.4f} (took {time.time()-start:.1f}s)")

# Convert to DataFrame for easy viewing
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("Hyperparameter Tuning Results")
print("="*60)
print(results_df.to_string(index=False))

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))
for n_est in n_estimators_values:
    subset = results_df[results_df['n_estimators'] == n_est]
    ax.plot(subset['max_depth'], subset['macro_f1'], marker='o', label=f'n_estimators={n_est}')

ax.set_xlabel('max_depth')
ax.set_ylabel('Macro F1 (Segment‑based)')
ax.set_title('Random Forest Hyperparameter Tuning')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('rf_hyperparameter_tuning.png', dpi=150)
plt.show()

# Identify best hyperparameters
best_row = results_df.loc[results_df['macro_f1'].idxmax()]
print(f"\nBest hyperparameters: max_depth={best_row['max_depth']}, n_estimators={int(best_row['n_estimators'])}")
print(f"Best Macro F1: {best_row['macro_f1']:.4f}")

In [ ]:
# ============================================================
# Question 2d: Qualitative Error Analysis
# ============================================================

import matplotlib.pyplot as plt

# Choose 2‑3 test files (you can change these)
test_filenames = [
    "000630.wav",   # from your earlier case study
    "001988.wav",   # from your earlier case study
    # Add a third file if desired
]

# Path to test features (hidden test set – but here we use validation test split features)
# We'll use the X_test_pca data, but we need the original .npz files to get the spectrogram.
# Since X_test_pca was derived from validation split, we need to map back to original files.

# Instead, we'll reload the original validation audio_features to get spectrograms
test_features_dir = val_path / "audio_features"

for fname in test_filenames:
    npz_path = test_features_dir / fname.replace('.wav', '.npz')
    if not npz_path.exists():
        print(f"File {npz_path} not found, skipping.")
        continue
    
    data = np.load(npz_path, allow_pickle=True)
    melspect = data['melspect_mean']  # shape [T, 128]
    
    # Find which indices in X_test correspond to this file (we need to map)
    # Simpler: since we have y_test and we know the file order, we could extract,
    # but for demo we can just plot spectrogram and show that we need true/pred.
    # For a proper analysis, you should save file-to-indices mapping during splitting.
    
    # Let's just plot the spectrogram as an example:
    plt.figure(figsize=(12, 6))
    plt.imshow(melspect.T, aspect='auto', origin='lower', cmap='magma',
               extent=[0, melspect.shape[0]*0.5, 0, 128])
    plt.colorbar(label='Magnitude')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Mel band')
    plt.title(f'Spectrogram of {fname}')
    plt.tight_layout()
    plt.savefig(f'spectrogram_{fname.replace(".wav","")}.png', dpi=150)
    plt.show()
    
    # For a full analysis, you need to align predictions with time segments.
    # You can use the same approach as in Task 4 case study.

In [ ]:
# ============================================================
# Question 3a: Post‑processing – Median Filtering
# ============================================================

from scipy.signal import medfilt

# ------------------------------
# 1. Get predictions from your best Random Forest model
# (Assuming rf_model is already trained and y_pred_test is available from 2a)
# ------------------------------

# If you haven't saved y_pred_test from 2a, re-run prediction:
# y_pred_test = rf_model.predict(X_test_pca)

# ------------------------------
# 2. Apply median filtering with different window sizes
# ------------------------------
window_sizes = [1, 3, 5, 7, 9, 11, 13, 15]  # 1 = no filtering
results_median = []

for w in window_sizes:
    print(f"Applying median filter with window size = {w}...")
    y_pred_filtered = np.zeros_like(y_pred_test)
    
    for c in range(n_classes):
        # medfilt expects 1D array, kernel size must be odd
        filtered = medfilt(y_pred_test[:, c], kernel_size=w)
        y_pred_filtered[:, c] = filtered
    
    # Compute Macro F1
    per_class_f1 = []
    for c in range(n_classes):
        f1 = f1_score(y_test[:, c], y_pred_filtered[:, c], zero_division=0)
        per_class_f1.append(f1)
    macro_f1 = np.mean(per_class_f1)
    results_median.append(macro_f1)
    print(f"   Macro F1 = {macro_f1:.4f}")

# ------------------------------
# 3. Plot results
# ------------------------------
plt.figure(figsize=(10, 6))
plt.plot(window_sizes, results_median, marker='o', linestyle='-', color='blue')
plt.axhline(y=results_median[0], color='red', linestyle='--', label=f'No filtering (w=1): {results_median[0]:.4f}')
plt.xlabel('Median Filter Window Size')
plt.ylabel('Segment‑based Macro F1')
plt.title('Effect of Median Filtering on Random Forest Predictions')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('median_filtering_results.png', dpi=150)
plt.show()

# Print best window size
best_idx = np.argmax(results_median)
print(f"\nBest window size: {window_sizes[best_idx]} with Macro F1 = {results_median[best_idx]:.4f}")
print(f"Improvement over no filtering: {results_median[best_idx] - results_median[0]:.4f}")

In [ ]:
# ============================================================
# Question 3b: Final Comparison (Baseline vs RF vs Post-processed)
# ============================================================

# ------------------------------
# 1. Get baseline results (already computed)
# ------------------------------
baseline_macro = 0.3038   # from your 1a output
baseline_per_class = [
    0.3333, 0.4247, 0.2974, 0.0703, 0.1791, 0.3646, 0.3370, 0.0000,
    0.4534, 0.5064, 0.6441, 0.3509, 0.5384, 0.0461, 0.0116
]

# ------------------------------
# 2. Get best Random Forest results (from 2a or 2b)
# ------------------------------
# If you haven't stored them, re-evaluate your best RF model
# Assuming rf_model is your trained best model from 2b
y_pred_rf = rf_model.predict(X_test_pca)
rf_per_class = []
for c in range(len(class_names)):
    rf_per_class.append(f1_score(y_test[:, c], y_pred_rf[:, c], zero_division=0))
rf_macro = np.mean(rf_per_class)

# ------------------------------
# 3. Get best post-processed results (from 3a)
# ------------------------------
# Use the best window size found in 3a (e.g., best_window)
best_window = 5   # replace with your actual best window size
y_pred_filtered = np.zeros_like(y_pred_rf)
from scipy.signal import medfilt
for c in range(n_classes):
    y_pred_filtered[:, c] = medfilt(y_pred_rf[:, c], kernel_size=best_window)

post_per_class = []
for c in range(len(class_names)):
    post_per_class.append(f1_score(y_test[:, c], y_pred_filtered[:, c], zero_division=0))
post_macro = np.mean(post_per_class)

# ------------------------------
# 4. Print comparison table
# ------------------------------
print("\n" + "="*70)
print("Final Comparison: Baseline vs Random Forest vs Post-processed")
print("="*70)
print(f"{'Class':<30} {'Baseline':>10} {'Random Forest':>15} {'Post-processed':>15}")
print("-"*70)
for i, name in enumerate(class_names):
    print(f"{name:<30} {baseline_per_class[i]:>10.4f} {rf_per_class[i]:>15.4f} {post_per_class[i]:>15.4f}")
print("-"*70)
print(f"{'Macro F1':<30} {baseline_macro:>10.4f} {rf_macro:>15.4f} {post_macro:>15.4f}")
print("="*70)

# Improvement over baseline
print(f"\nRandom Forest improvement: +{rf_macro - baseline_macro:.4f}")
print(f"Post-processed improvement: +{post_macro - baseline_macro:.4f}")